# Clone and test Fv energy analysis on two complexes

Run this notebook with a **Windows Python kernel**. The controller runs native
Discovery Studio; its scientific subprocesses use the configured WSL environment.
Copy the example settings JSON, set your input paths, and point SETTINGS_PATH at it.
The repository is cloned from GitHub and its exact commit is recorded.

Set optimize_hydrogens=true (the default) for one whole-complex OptH pass on the
parent and both candidates before analysis. Set it to false to analyze supplied
coordinates directly, for example already optimized PDBs. All detectors and
reports use the selected structures. The notebook recalculates RosettaLigand scores, Arpeggio,
PLIP, DS, networks, RMSDs/rotamers, E2/E3 features and supplied metadata in a separate
output folder. Existing scientific dependencies are reused; old analysis results
and parent caches are not imported.


In [ ]:
import csv
import hashlib
import importlib
import json
import os
from pathlib import Path
import shlex
import shutil
import subprocess
import sys
import time
from datetime import datetime, timezone

SETTINGS_PATH = Path(os.environ.get("FV_NOTEBOOK_SETTINGS", "two_complex_settings.json")).resolve()
if os.name != "nt":
    raise RuntimeError("Use a Windows kernel for native Discovery Studio and the full E2/E3 analysis.")
settings = json.loads(SETTINGS_PATH.read_text(encoding="utf-8-sig"))
run_root = Path(settings["run_root"]).resolve()
optimize_hydrogens = settings.get("optimize_hydrogens", True)
if not isinstance(optimize_hydrogens, bool):
    raise ValueError("optimize_hydrogens must be true or false")
prior_status = run_root / "analysis/full_pipeline_status.json"
if prior_status.is_file():
    prior = json.loads(prior_status.read_text(encoding="utf-8-sig"))
    recorded = prior.get("scientific_options", {}).get("optimize_hydrogens", True)
    if recorded != optimize_hydrogens:
        raise ValueError("OptH setting changed. Choose a new run_root; existing results were not modified.")
run_root.mkdir(parents=True, exist_ok=True)
logs = run_root / "logs"
logs.mkdir(exist_ok=True)
clone = run_root / "cloned_repository"

def run_command(arguments, name, *, cwd=None, env=None):
    log = logs / (name + ".log")
    started = time.perf_counter()
    print(f"{name}: started", flush=True)
    with log.open("w", encoding="utf-8") as stream:
        result = subprocess.run([str(x) for x in arguments], cwd=cwd, env=env,
                                stdout=stream, stderr=subprocess.STDOUT, text=True)
    tail = log.read_text(encoding="utf-8", errors="replace").splitlines()[-20:]
    print("\n".join(tail))
    print(f"{name}: exit={result.returncode}; {time.perf_counter()-started:.1f}s; log={log}")
    if result.returncode:
        raise RuntimeError(f"{name} failed; inspect {log}")
    return log

def digest(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8-sig"))

print("Kernel:", sys.executable)
print("Settings:", SETTINGS_PATH)


## Clone the public repository and install the controller package


In [ ]:
if not clone.exists():
    run_command(["git", "clone", settings["repository"], clone], "clone")
actual_origin = subprocess.check_output(
    ["git", "-C", str(clone), "remote", "get-url", "origin"], text=True).strip()
assert actual_origin.removesuffix(".git") == settings["repository"].removesuffix(".git")
if settings.get("revision"):
    present = subprocess.run(["git", "-C", str(clone), "cat-file", "-e",
                              settings["revision"] + "^{commit}"], capture_output=True)
    if present.returncode:
        run_command(["git", "-C", clone, "fetch", "origin", settings["revision"]], "fetch_revision")
    run_command(["git", "-C", clone, "checkout", "--detach", settings["revision"]], "checkout")
commit = subprocess.check_output(["git", "-C", str(clone), "rev-parse", "HEAD"], text=True).strip()
assert not subprocess.check_output(["git", "-C", str(clone), "status", "--porcelain"], text=True).strip()
run_command([sys.executable, "-m", "pip", "install",
             str(clone) + "[reporting]"], "install_controller")
sys.path.insert(0, str(clone / "src"))
importlib.invalidate_caches()
import fv_energy_analysis
assert Path(fv_energy_analysis.__file__).resolve().is_relative_to(clone)
child_environment = os.environ.copy()
child_environment["PYTHONPATH"] = str(clone / "src")
child_environment["PYTHONUNBUFFERED"] = "1"
print("Git commit:", commit)
print("Windows package:", fv_energy_analysis.__file__)


## Check the existing scientific environment

The scientific dependencies, including PyRosetta, must already be installed.
For a first installation, use scripts/bootstrap_wsl.sh as documented in the
repository. This test does not upgrade that environment or change its editable
installation. PYTHONPATH selects the fresh clone in each scientific subprocess.


In [ ]:
def to_wsl(path):
    path = Path(path).resolve()
    return "/mnt/" + path.drive[0].lower() + "/" + "/".join(path.parts[1:])

science_python = settings["wsl_python"]
proof = (
    "import json,fv_energy_analysis,sys; "
    "print(json.dumps({'package':fv_energy_analysis.__file__,'python':sys.executable}))"
)
command = (
    "export PYTHONPATH=" + shlex.quote(to_wsl(clone / "src")) + "; exec "
    + shlex.join([science_python, "-c", proof])
)
proof_log = run_command(["wsl.exe", "-d", settings["distro"], "--",
                         "/bin/bash", "-c", command], "science_import")
science_import = json.loads(proof_log.read_text().strip().splitlines()[-1])
assert science_import["package"].startswith(to_wsl(clone / "src") + "/")
print("WSL package:", science_import["package"])


## Stage exactly two complexes and their matching metadata


In [ ]:
candidates = [Path(p).resolve() for p in settings["candidates"]]
assert len(candidates) == 2 and len({p.name for p in candidates}) == 2
parent = Path(settings["parent"]).resolve()
inputs = run_root / "inputs"
mutants = inputs / "mutants"
mutants.mkdir(parents=True, exist_ok=True)
source_hashes = {}
staged_hashes = {}

def stage_file(source, destination):
    source, destination = Path(source).resolve(), Path(destination)
    assert source.is_file(), source
    expected = digest(source)
    source_hashes[str(source)] = expected
    if destination.is_file():
        assert digest(destination) == expected, destination
    else:
        shutil.copy2(source, destination)
    staged_hashes[str(destination)] = expected
    return str(destination.resolve())

staged_candidates = [stage_file(p, mutants / p.name) for p in candidates]
assert sorted(p.name for p in mutants.glob("*.pdb")) == sorted(p.name for p in candidates)
config = {
    "workspace": str(clone), "science_backend": "wsl",
    "distro": settings["distro"], "wsl_python": science_python,
    "parent": stage_file(parent, inputs / "parent_m7.pdb"),
    "input_folder": str(mutants), "output": str(run_root / "analysis"),
    "params": stage_file(settings["params"], inputs / "E2L.params"),
    "ligand_template_pdb": stage_file(settings["ligand_template_pdb"], inputs / "E2L_0001.pdb"),
    "discovery_studio_root": settings["discovery_studio_root"],
    "jobs": 2, "weights": "ligand", "legacy_pre_talaris": True,
    "optimize_hydrogens": optimize_hydrogens, "expected_count": 2,
    "ligand_chain": "X", "parent_ligand_chain": "X", "mutant_ligand_chain": "X",
    "include_empty_metadata_sections": True,
    "scheme": "imgt", "cdr_definition": "imgt", "fit_region": "fv",
    "rmsd_atoms": "backbone", "rotamer_rmsd_cutoff": 3.0,
    "hcdr3_tip_sequence": "YYGSS", "arpeggio_scope": "neighbourhood",
    "arpeggio_neighbourhood_radius": 6.0, "input_glob": "*.pdb",
    "analyses": {key: True for key in (
        "arpeggio", "plip", "discovery_studio", "e2_architecture",
        "e3_discrimination", "network", "rotamers", "hcdr3_plddt",
        "boltz2_scores", "gnina_scores")},
    "e3_overlap_warning": 0.4, "report_formats": ["csv", "xlsx"],
}
if optimize_hydrogens:
    config["hydrogen_output"] = str(run_root / "analysis_opth")
    config["hydrogen_seed"] = settings.get("hydrogen_seed", 20261022)
cifs = inputs / "cifs"
cifs.mkdir(exist_ok=True)
assert len(settings["candidate_cifs"]) == 2
for value in settings["candidate_cifs"]:
    stage_file(value, cifs / Path(value).name)
config["cif_folder"] = str(cifs)
config["parent_cif"] = stage_file(settings["parent_cif"], cifs / "parent_m7.cif")
config["scores_csv"] = stage_file(settings["scores_csv"], inputs / "model_scores.csv")
for key in ("params", "ligand_template_pdb"):
    config[key + "_sha256"] = digest(config[key])
config_path = run_root / "pipeline_config.json"
prior = Path(config["output"]) / "full_pipeline_status.json"
fresh_run = not prior.exists()
if not fresh_run and not settings.get("resume", False):
    raise FileExistsError("This test already has results. Set resume=true to verify/reuse them, or use a new run_root.")
if not fresh_run:
    saved_batch = Path(config["output"]) / "batch_manifest.json"
    if saved_batch.is_file() and read_json(saved_batch).get("status") == "complete":
        config["postprocessing_only"] = True
config_path.write_text(json.dumps(config, indent=2) + "\n", encoding="utf-8")
print("Fresh analysis:", fresh_run)
print("Config:", config_path)
print("Candidates:", [p.name for p in candidates])


## Preflight, then run the normal module once


In [ ]:
run_command([sys.executable, "-u", "-m", "fv_energy_analysis.full_pipeline",
             config_path, "--preflight"], "preflight", cwd=clone, env=child_environment)


In [ ]:
pipeline_started = time.perf_counter()
run_command([sys.executable, "-u", "-m", "fv_energy_analysis.full_pipeline", config_path],
            "full_pipeline", cwd=clone, env=child_environment)
pipeline_seconds = time.perf_counter() - pipeline_started


## Verify every stage, file, metadata join and workbook cell


In [ ]:
from fv_energy_analysis.paths import native_path
from fv_energy_analysis.workbook_validation import verify

batch = Path(config["output"])
status = read_json(batch / "full_pipeline_status.json")
assert status["status"] == "complete", status
assert status["completed_count"] == 2 and status["scientific_failure_count"] == 0
expected_stages = {"scientific_analysis", "discovery_studio", "fingerprints_and_scores",
                   "e3_source_inventory", "e3_steric_discrimination", "excel_export"}
completed = {name.removesuffix("_reused") for name in status["completed_stages"]}
if optimize_hydrogens:
    expected_stages.add("hydrogen_optimization")
else:
    assert "hydrogen_optimization" not in completed
    assert "hydrogen_output" not in config
assert expected_stages <= completed
opth = None
if optimize_hydrogens:
    opth_folder = Path(config["hydrogen_output"])
    opth = read_json(opth_folder / "hydrogen_batch_manifest.json")
    assert opth["status"] == "complete"
    assert (opth["candidate_count"], opth["completed_count"], opth["failure_count"]) == (2, 2, 0)
    assert opth["protocol"]["passes_per_complex"] == 1
    assert opth["protocol"]["source_protocol"]["weights"] == "ligand"
    assert opth["protocol"]["source_protocol"]["legacy_pre_talaris"] is True
    records = [opth_folder / "parent/result.json"]
    records += [opth_folder / "cases" / Path(p).stem / "result.json" for p in staged_candidates]
    for record_path in records:
        record = read_json(record_path)
        assert record["status"] == "complete"
        assert record["identity"]["protocol"] == opth["protocol"]
        assert record["optimization"]["persisted_audit"]["max_heavy_displacement_A"] <= 0.002
        for artifact, expected_hash in record["artifact_sha256"].items():
            assert digest(native_path(artifact)) == expected_hash
manifest = read_json(batch / "batch_manifest.json")
assert (manifest["candidate_count"], manifest["completed_count"], manifest["failure_count"]) == (2, 2, 0)
assert manifest["optimization"]["parent_scored_once"] is True
assert len(manifest["candidates"]) == 2
assert all(c["status"] == "complete" for c in manifest["candidates"])
if fresh_run:
    assert all(c["reused"] is False for c in manifest["candidates"])
ds = manifest["discovery_studio"]
assert ds["status"] == "complete" and ds["completed_structures"] == 3
assert ds["add_hydrogens"] is False
assert manifest["hcdr3_confidence"]["matched_candidates"] == 2
assert manifest["model_scores"]["matched_candidates"] == 2
checks = []
for candidate in manifest["candidates"]:
    case = native_path(candidate["result_folder"])
    workflow = read_json(case / "workflow_manifest.json")
    assert workflow["status"] == "complete"
    assert all(workflow["analysis_status"][k] == "complete"
               for k in ("arpeggio", "plip", "network", "rotamers", "e2_architecture"))
    scoring = workflow["scoring_protocol"]
    assert scoring["weights"] == "ligand" and scoring["legacy_pre_talaris"] is True
    assert scoring["weights_identity"]["weights_sha256"]
    if optimize_hydrogens:
        optimized = opth_folder / "hydrogen_optimized_structures"
        expected_parent = optimized / "parent_m7.pdb"
        expected_mutant = optimized / Path(candidate["candidate"]).name
        assert scoring["weights_identity"]["weights_sha256"] == opth["protocol"]["source_protocol"]["weights_sha256"]
    else:
        expected_parent = Path(config["parent"])
        expected_mutant = mutants / Path(candidate["candidate"]).name
    assert digest(native_path(workflow["parent"])) == digest(expected_parent)
    assert digest(native_path(workflow["mutant"])) == digest(expected_mutant)
    for filename in (
        "arpeggio_mutant/mutant_arpeggio.json", "plip_mutant/mutant_plip.xml",
        "discovery_studio_interactions.csv", "discovery_studio_interactions.json",
        "mutant_interface.sc", "mutant_pair_energies.sc", "mutant_hbonds.csv",
        "final/mutation_network_changes.csv", "final/mutation_network_type_energy_summary.csv",
        "final/substitutions.csv", "final/structure_rmsd.csv", "automatic_summary.md",
        "hcdr3_plddt.json", "model_scores.json",
    ):
        artifact = case / filename
        assert artifact.is_file() and artifact.stat().st_size > 0, artifact
    chemistry = read_json(case / "ligand_atom_validation.json")
    architecture = read_json(case / "estrogen_binding_architecture.json")
    for role in ("parent", "mutant"):
        assert chemistry[role]["valid"] is True
        assert chemistry[role]["status"] == "complete_explicit_hydrogens"
        assert architecture[role]["e2_architecture_status"] == "complete"
    checks.append({"candidate": Path(candidate["candidate"]).name,
                   "status": "passed", "result_folder": str(case),
                   "weights_sha256": scoring["weights_identity"]["weights_sha256"]})

final_folder = Path(status["final_report_folder"])
e3 = read_json(final_folder / "e3_batch_manifest.json")
assert e3["status"] == "complete" and e3["candidate_count"] == 2
assert e3["status_counts"] == {"complete": 2}
payload = read_json(final_folder / "batch_summary_grouped_fingerprints.json")
assert len(payload["rows"]) == 2
keys = [column["key"] for column in payload["columns"]]
with (final_folder / "batch_summary_grouped_fingerprints.csv").open(
        encoding="utf-8-sig", newline="") as stream:
    reader = csv.DictReader(stream)
    rows = list(reader)
    assert reader.fieldnames == keys
assert len(rows) == 2 and [r["candidate"] for r in rows] == [r["candidate"] for r in payload["rows"]]
workbook_check = verify(final_folder)
assert workbook_check["status"] == "verified"
assert workbook_check["candidate_count"] == 2 and workbook_check["failure_count"] == 0
assert all(digest(p) == h for p, h in source_hashes.items()), "A source input changed"
assert all(digest(p) == h for p, h in staged_hashes.items()), "A staged input changed"
evidence = {
    "status": "passed", "checked_utc": datetime.now(timezone.utc).isoformat(),
    "repository": settings["repository"], "commit": commit,
    "scientific_source_commit": settings.get("scientific_source_revision", commit),
    "kernel": sys.executable, "windows_package": fv_energy_analysis.__file__,
    "wsl_import": science_import, "fresh_controller_run": fresh_run,
    "fresh_scientific_results": all(c["reused"] is False for c in manifest["candidates"]),
    "postprocessing_only_resume": config.get("postprocessing_only", False),
    "optimize_hydrogens": optimize_hydrogens,
    "hydrogens": ("one whole-complex OptH pass before all analyses" if optimize_hydrogens
                  else "supplied coordinates; OptH disabled"),
    "hydrogen_output": config.get("hydrogen_output"),
    "source_sha256": source_hashes, "staged_sha256": staged_hashes,
    "pipeline_seconds": round(pipeline_seconds, 2), "cases": checks,
    "completed_stages": status["completed_stages"], "workbook": workbook_check,
    "final_csv": status["final_csv"], "final_report_folder": str(final_folder),
}
(run_root / "notebook_validation.json").write_text(json.dumps(evidence, indent=2) + "\n")
print(json.dumps(evidence, indent=2))


## Results


In [ ]:
from IPython.display import display, Markdown, FileLink

view = ("candidate", "mutations", "rosetta_interface_REU",
        "rosetta_interface_delta_REU", "e2_architecture_status",
        "e2_strict_screen_pass")
available = [k for k in view if k in keys]
text = "| " + " | ".join(available) + " |\n"
text += "| " + " | ".join("---" for _ in available) + " |\n"
for row in payload["rows"]:
    text += "| " + " | ".join(str(row.get(k, "")) for k in available) + " |\n"
display(Markdown(text))
display(FileLink(str(final_folder / "batch_summary_grouped_fingerprints.xlsx")))
display(FileLink(str(final_folder / "batch_summary_grouped_fingerprints.csv")))
print("PASS: both complexes completed; all requested analyses and metadata were verified.")
